In [13]:
!pip install -q -U google-genai sentence-transformers faiss-cpu pymupdf gradio

In [2]:
import os
import re
import hashlib
import tempfile
import pymupdf as fitz
import numpy as np
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import userdata

print("Modules importés.")

Modules importés.


In [3]:
# Vérification réelle de la clé, des modèles accessibles et d'une génération test.
client = None
available_gemini_model = None
GOOGLE_API_KEY = None
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    if not GOOGLE_API_KEY:
        print("Secret GOOGLE_API_KEY absent : la génération ne pourra pas être testée.")
    else:
        client = genai.Client(api_key=GOOGLE_API_KEY)
        listed = list(client.models.list())
        model_names = [m.name.replace("models/", "") for m in listed if getattr(m, "name", None)]
        # On ne suppose pas la disponibilité : on teste les candidats présents dans la liste.
        candidates = [name for name in model_names if "gemini" in name.lower() and "embedding" not in name.lower()]
        for candidate in candidates:
            try:
                test_response = client.models.generate_content(model=candidate, contents="Réponds simplement : test OK.")
                if getattr(test_response, "text", None):
                    available_gemini_model = candidate
                    print(f"Test de génération réel réussi avec : {candidate}")
                    break
            except Exception as test_error:
                print(f"Test non concluant pour {candidate} : {type(test_error).__name__}: {test_error}")
        if available_gemini_model is None:
            print("Aucun modèle listé n'a réussi le test de génération. Vérifiez la clé, le quota et les modèles disponibles.")
except Exception as e:
    print(f"Configuration/test LLM non effectué ou en échec : {type(e).__name__}: {e}")
    client = None
    available_gemini_model = None

Configuration/test LLM non effectué ou en échec : TimeoutException: Requesting secret GOOGLE_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.


In [4]:
def extract_text_from_pdf(pdf_path, source_name=None):
    """Extraction page par page, fermeture garantie et métadonnées fichier/page."""
    documents_data = []
    filename = source_name or os.path.basename(str(pdf_path))
    doc = None
    try:
        doc = fitz.open(str(pdf_path))
        for page_num in range(len(doc)):
            page_text = doc.load_page(page_num).get_text("text")
            if page_text and page_text.strip():
                documents_data.append({"text": page_text.strip(),
                    "metadata": {"source": filename, "page": page_num + 1}})
    except Exception as e:
        print(f"Erreur PDF {filename}: {type(e).__name__}: {e}")
    finally:
        if doc is not None:
            doc.close()
    return documents_data

In [5]:
def clean_text(text):
    return re.sub(r'\s+', ' ', text or '').strip()

def create_chunks(extracted_docs, chunk_size=600, chunk_overlap=100, start_id=0):
    """Découpe avec chevauchement; progression garantie et chunks non vides."""
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size doit être un entier positif.")
    chunk_overlap = int(chunk_overlap)
    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap doit être compris entre 0 et chunk_size-1.")
    chunks, chunk_id = [], start_id
    step = chunk_size - chunk_overlap
    for doc in extracted_docs:
        cleaned = clean_text(doc.get("text", ""))
        n = len(cleaned)
        start = 0
        while start < n:
            end = min(start + chunk_size, n)
            text_piece = cleaned[start:end].strip()
            if text_piece:
                chunks.append({"chunk_id": chunk_id, "text": text_piece,
                               "metadata": dict(doc.get("metadata", {}))})
                chunk_id += 1
            if end >= n:
                break
            start += step
    return chunks

In [6]:
# Chargement du modèle d'embeddings du retriever
print("Chargement du modèle d'embeddings...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = int(embedder.get_sentence_embedding_dimension())

class RAGIndex:
    def __init__(self, dimension=EMBEDDING_DIM):
        self.dimension = int(dimension)
        self.index = faiss.IndexFlatIP(self.dimension)
        self.chunks_store = []
        self.indexed_files = set()  # SHA-256 du contenu, indépendant du nom de fichier

    def add_chunks(self, chunks, file_key=None):
        if not chunks or (file_key and file_key in self.indexed_files):
            return 0
        valid_chunks = [c for c in chunks if isinstance(c.get("text"), str) and c["text"].strip()]
        if not valid_chunks:
            return 0
        texts = [c["text"] for c in valid_chunks]
        embeddings = np.asarray(embedder.encode(texts, convert_to_numpy=True), dtype=np.float32)
        if embeddings.ndim != 2 or embeddings.shape[0] != len(valid_chunks) or embeddings.shape[1] != self.dimension:
            raise ValueError(f"Embeddings incompatibles avec l'index : shape={embeddings.shape}, dimension={self.dimension}.")
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
        self.chunks_store.extend(valid_chunks)
        if file_key: self.indexed_files.add(file_key)
        if self.index.ntotal != len(self.chunks_store):
            raise RuntimeError("Désynchronisation FAISS / chunks.")
        return len(valid_chunks)

rag_index = RAGIndex()

Chargement du modèle d'embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1304/2337900853.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = int(embedder.get_sentence_embedding_dimension())


In [7]:
def search_index(query, k=3):
    if not isinstance(query, str) or not query.strip() or not rag_index.chunks_store:
        return []
    try:
        k = int(k)
    except (TypeError, ValueError):
        return []
    if k <= 0 or rag_index.index.ntotal == 0:
        return []
    if rag_index.index.ntotal != len(rag_index.chunks_store):
        raise RuntimeError("Index FAISS et stockage des chunks désynchronisés.")
    k = min(k, rag_index.index.ntotal)
    qvec = np.asarray(embedder.encode([query.strip()], convert_to_numpy=True), dtype=np.float32)
    if qvec.ndim != 2 or qvec.shape[0] != 1 or qvec.shape[1] != rag_index.dimension:
        raise ValueError("Dimension du vecteur de requête incompatible avec FAISS.")
    faiss.normalize_L2(qvec)
    scores, indices = rag_index.index.search(qvec, k)
    return [{"chunk": rag_index.chunks_store[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])
            if 0 <= idx < len(rag_index.chunks_store)]

In [8]:
def generate_answer_with_rag(query, k=3):
    if not isinstance(query, str) or not query.strip():
        return "Veuillez saisir une question.", []
    if client is None or available_gemini_model is None:
        return "Le générateur LLM n'est pas configuré ou n'a pas réussi son test réel. Le retriever RAG peut être évalué séparément.", []
    results = search_index(query, k=k)
    if not results:
        return "Aucun passage pertinent récupéré. Indexez des PDF et vérifiez la question.", []
    context_parts, sources = [], []
    for i, result in enumerate(results, 1):
        chunk = result["chunk"]
        source = f"{chunk['metadata'].get('source', 'Inconnu')} (page {chunk['metadata'].get('page', '?')})"
        context_parts.append(f"[Passage {i} | Source : {source}]\n{chunk['text']}")
        sources.append(source)
    prompt = (
        "Tu es le composant génératif d'un système RAG. Réponds en français, uniquement à partir des passages ci-dessous. "
        "N'utilise pas de connaissances externes. Si la réponse n'est pas présente, indique clairement que les documents ne permettent pas de répondre. "
        "Ne prétends pas avoir consulté d'autres sources.\n\n" + "\n\n".join(context_parts) + "\n\nQuestion : " + query.strip()
    )
    try:
        response = client.models.generate_content(model=available_gemini_model, contents=prompt)
        answer = getattr(response, "text", None) or "Le modèle n'a retourné aucun texte."
        return answer, list(dict.fromkeys(sources))
    except Exception as e:
        code = getattr(e, "status_code", None) or getattr(getattr(e, "response", None), "status_code", None)
        if code == 404: msg = "Erreur 404 : modèle ou endpoint non disponible."
        elif code == 429: msg = "Erreur 429 : quota ou limite de requêtes dépassée."
        elif code == 503: msg = "Erreur 503 : service temporairement indisponible."
        else: msg = f"Erreur du générateur ({code or type(e).__name__}) : {e}"
        return msg, list(dict.fromkeys(sources))

In [9]:
def _file_path_and_name(file_obj):
    """Compatibilité avec chemins Gradio, objets File et dictionnaires."""
    if isinstance(file_obj, dict):
        path = file_obj.get("path") or file_obj.get("filepath") or file_obj.get("name")
        display_name = file_obj.get("orig_name") or file_obj.get("filename") or os.path.basename(str(path))
    else:
        path = getattr(file_obj, "path", None) or getattr(file_obj, "name", None) or str(file_obj)
        display_name = getattr(file_obj, "orig_name", None) or os.path.basename(str(path))
    if not path or not os.path.isfile(str(path)):
        raise ValueError("Chemin de fichier PDF introuvable ou non accessible.")
    return str(path), str(display_name)

def process_pdf_and_index(file_objs):
    if not file_objs: return "Aucun PDF fourni."
    files = file_objs if isinstance(file_objs, (list, tuple)) else [file_objs]
    added_files, added_chunks, skipped, errors = 0, 0, 0, []
    for file_obj in files:
        try:
            filepath, filename = _file_path_and_name(file_obj)
            with open(filepath, "rb") as f: digest = hashlib.sha256(f.read()).hexdigest()
            if digest in rag_index.indexed_files:
                skipped += 1; continue
            extracted = extract_text_from_pdf(filepath, source_name=filename)
            chunks = create_chunks(extracted, chunk_size=600, chunk_overlap=100, start_id=len(rag_index.chunks_store))
            count = rag_index.add_chunks(chunks, file_key=digest)
            if count:
                added_chunks += count; added_files += 1
            else:
                errors.append(f"{filename}: aucun texte exploitable")
        except Exception as e:
            errors.append(f"Fichier : {type(e).__name__}: {e}")
    status = f"Indexation : {added_files} fichier(s), {added_chunks} chunk(s) ajouté(s). Total : {len(rag_index.chunks_store)} chunk(s). Fichiers identiques ignorés : {skipped}."
    if errors: status += "\nErreurs : " + " | ".join(errors)
    return status

def gradio_ask(query):
    answer, sources = generate_answer_with_rag(query)
    return answer, "\n".join(f"- {s}" for s in sources) if sources else "Aucune source récupérée."

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 RAG Intelligent PDF Assistant")
    gr.Markdown("**Pipeline RAG :** documents → embeddings → recherche FAISS → passages récupérés → réponse contextualisée. Chargez vos PDF puis posez une question.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Étape 1 — Documents")
            file_input = gr.File(label="PDF", file_count="multiple", file_types=[".pdf"], type="filepath")
            upload_btn = gr.Button("Indexer les documents", variant="primary")
            status_output = gr.Textbox(label="Statut de l'indexation", interactive=False)
        with gr.Column(scale=2):
            gr.Markdown("### Étape 2 — Question")
            query_input = gr.Textbox(label="Question", placeholder="Posez une question sur vos documents...")
            ask_btn = gr.Button("Rechercher et répondre")
            answer_output = gr.Textbox(label="Réponse basée sur les passages récupérés", lines=8, interactive=False)
            sources_output = gr.Textbox(label="Sources (fichier et page)", lines=4, interactive=False)
    upload_btn.click(process_pdf_and_index, inputs=file_input, outputs=status_output)
    ask_btn.click(gradio_ask, inputs=query_input, outputs=[answer_output, sources_output])

print("Interface définie. Dans Colab, lancez demo.launch(inline=True, share=False).")
demo.launch(inline=True, share=False)

/tmp/ipykernel_1304/694724091.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Interface définie. Dans Colab, lancez demo.launch(inline=True, share=False).
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [10]:
def evaluate_retrieval(evaluation_dataset, k=3):
    if not rag_index.chunks_store:
        return "Index vide : indexez des PDF avant l'évaluation."
    if not evaluation_dataset:
        return "Aucun jeu de test valide fourni. Complétez mon_jeu_de_test avec des questions et les IDs réels des chunks pertinents."
    try: k = int(k)
    except (TypeError, ValueError): return "K doit être un entier positif."
    if k <= 0: return "K doit être un entier positif."
    valid_ids = {c.get("chunk_id") for c in rag_index.chunks_store if isinstance(c.get("chunk_id"), int)}
    precisions, recalls, reciprocal_ranks = [], [], []
    for item in evaluation_dataset:
        query = item.get("query", "") if isinstance(item, dict) else ""
        expected = item.get("expected_chunk_ids", []) if isinstance(item, dict) else []
        if not isinstance(query, str) or not query.strip() or not isinstance(expected, list) or not expected:
            continue
        if not all(isinstance(x, int) and x in valid_ids for x in expected):
            continue
        expected_ids = set(expected)
        results = search_index(query, k=k)
        retrieved_ids = [r["chunk"]["chunk_id"] for r in results]
        hits = len(set(retrieved_ids) & expected_ids)
        precisions.append(hits / len(retrieved_ids) if retrieved_ids else 0.0)
        recalls.append(hits / len(expected_ids) if expected_ids else 0.0)
        reciprocal_ranks.append(next((1.0/(rank+1) for rank, rid in enumerate(retrieved_ids) if rid in expected_ids), 0.0))
    if not precisions:
        return "Aucun exemple valide : fournissez query et expected_chunk_ids correspondant aux IDs réellement présents dans l'index."
    return {"Precision@K": float(np.mean(precisions)), "Recall@K": float(np.mean(recalls)), "MRR": float(np.mean(reciprocal_ranks))}

# Dataset vide par défaut : ajoutez des questions et les IDs pertinents constatés après indexation.
mon_jeu_de_test = []
print(evaluate_retrieval(mon_jeu_de_test, k=3))

Index vide : indexez des PDF avant l'évaluation.


In [11]:
# Contrôles rapides du notebook (à exécuter après les cellules correspondantes)
# Ces assertions ne simulent pas des résultats Gemini ou FAISS. Elles valident les fonctions locales.
assert clean_text("  texte\n test  ") == "texte test"
_test_chunks = create_chunks([{"text": "x" * 1300, "metadata": {"source": "test.pdf", "page": 1}}])
assert _test_chunks and all(c["text"] for c in _test_chunks)
assert all(c["metadata"]["page"] == 1 for c in _test_chunks)
assert len(_test_chunks) > 1
print("Contrôles locaux du nettoyage et chunking : OK")

Contrôles locaux du nettoyage et chunking : OK
